# Logistic Regression with L1 and L2 Regularization

In [97]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score


In [98]:
df = pd.read_csv("../data/drug_discovery_virtual_screening_cleaned.csv")
# df = df.drop(['logp_pi_interaction'], axis=1) 

X = df.drop(columns=["active"])
y = df["active"]
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, stratify=y, random_state=42
)

In [99]:
log_pipeline = Pipeline([
  ("scaler", StandardScaler()),
  ("model", LogisticRegression(max_iter=10000, solver="liblinear"))
])

## L2 Regularization (Ridge)

In [100]:
l2_params = {
  "model__C": [0.01, 0.1, 1, 10],
  "model__penalty": ["l2"]
}

l2_grid = GridSearchCV(
  log_pipeline,
  l2_params,
  cv=5,
  scoring="roc_auc",
  n_jobs=-1
)

l2_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >

In [101]:
l2_best = l2_grid.best_estimator_
l2_preds = l2_best.predict(X_test)
l2_probs = l2_best.predict_proba(X_test)[:, 1]

print("=== L2 Logistic Regression (Tuned) ===")
print("Best Params:", l2_grid.best_params_)
print(classification_report(y_test, l2_preds))
print("ROC-AUC:", roc_auc_score(y_test, l2_probs))

=== L2 Logistic Regression (Tuned) ===
Best Params: {'model__C': 10, 'model__penalty': 'l2'}
              precision    recall  f1-score   support

         0.0       0.90      0.95      0.92       278
         1.0       0.87      0.76      0.81       122

    accuracy                           0.89       400
   macro avg       0.89      0.86      0.87       400
weighted avg       0.89      0.89      0.89       400

ROC-AUC: 0.9577190706451233


## L1 Regularization (Lasso)

In [102]:
l1_params = {
  "model__C": [0.01, 0.1, 1, 10],
  "model__penalty": ["l1"]
}

l1_grid = GridSearchCV(
  log_pipeline,
  l1_params,
  cv=5,
  scoring="roc_auc",
  n_jobs=-1
)

l1_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l1']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >

In [103]:
l1_best = l1_grid.best_estimator_
l1_preds = l1_best.predict(X_test)
l1_probs = l1_best.predict_proba(X_test)[:, 1]

print("=== L1 Logistic Regression (Tuned) ===")
print("Best Params:", l1_grid.best_params_)
print(classification_report(y_test, l1_preds))
print("ROC-AUC:", roc_auc_score(y_test, l1_probs))

=== L1 Logistic Regression (Tuned) ===
Best Params: {'model__C': 0.01, 'model__penalty': 'l1'}
              precision    recall  f1-score   support

         0.0       0.90      0.95      0.92       278
         1.0       0.86      0.76      0.81       122

    accuracy                           0.89       400
   macro avg       0.88      0.85      0.87       400
weighted avg       0.89      0.89      0.89       400

ROC-AUC: 0.954475763651374


## Feature survival for L1

In [104]:
l1_model = l1_best.named_steps["model"]
coef = pd.Series(l1_model.coef_[0], index=X.columns)
print("L1 coefficients (sorted):")
print(coef.sort_values())
print(f"\nNonzero features: {(coef != 0).sum()} of {len(coef)}")

L1 coefficients (sorted):
Unnamed: 0             0.000000
molecular_weight       0.000000
logp                   0.000000
h_bond_donors          0.000000
h_bond_acceptors       0.000000
rotatable_bonds        0.000000
polar_surface_area     0.000000
compound_clogp         0.000000
protein_length         0.000000
protein_pi             0.000000
hydrophobicity         0.000000
binding_site_size      0.000000
mw_ratio               0.000000
logp_pi_interaction    1.569481
dtype: float64

Nonzero features: 1 of 14


## Threshold tuning


In [105]:
y_prob = l2_best.predict_proba(X_test)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91) #91 steps is step size of 0.01 here
rows = []
for t in thresholds:
    preds = (y_prob >= t)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds)
    })

pr_table = pd.DataFrame(rows)
pr_table.head()

,threshold,precision,recall,f1
0,0.05,0.510638,0.983607,0.672269
1,0.06,0.538117,0.983607,0.695652
2,0.07,0.560748,0.983607,0.714286
3,0.08,0.581281,0.967213,0.726154
4,0.09,0.590000,0.967213,0.732919


### Precision and Recall vs Threshold

In [106]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=pr_table["threshold"], y=pr_table["precision"], name="precision"))
fig.add_trace(go.Scatter(x=pr_table["threshold"], y=pr_table["recall"], name="recall"))
fig.add_trace(go.Scatter(x=pr_table["threshold"], y=pr_table["f1"], name="F1"))
fig.add_vline(x=0.5, line_dash="dot", line_color="gray", annotation_text="default (0.5)")
fig.update_layout(
    title="Precision, Recall, and F1 vs decision threshold",
    xaxis_title="threshold",
    yaxis_title="score")
fig.show()

### Best 

In [108]:
default_row = pr_table[np.isclose(pr_table["threshold"], 0.5)].iloc[0]
best_f1_row = pr_table.loc[pr_table["f1"].idxmax()]
best_recall_row = pr_table[pr_table["recall"] >= 0.90].sort_values("threshold").iloc[-1]

summary = pd.DataFrame([default_row, best_f1_row, best_recall_row])
summary.insert(0, "label", ["default", "best F1", "recall >= 0.90"])
summary.round(3)

,label,threshold,precision,recall,f1
45,default,0.50,0.869,0.762,0.812
27,best F1,0.32,0.835,0.869,0.851
20,recall >= 0.90,0.25,0.764,0.902,0.827


In [109]:
for label, t in [("Default (0.5)", 0.5), ("Best F1", best_f1_row["threshold"]), ("Recall>=0.90", best_recall_row["threshold"])]:
    preds = (y_prob >= t)
    print(f"=== {label} (threshold = {t}) ===")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))
    print()

=== Default (0.5) (threshold = 0.5) ===
[[264  14]
 [ 29  93]]
              precision    recall  f1-score   support

         0.0       0.90      0.95      0.92       278
         1.0       0.87      0.76      0.81       122

    accuracy                           0.89       400
   macro avg       0.89      0.86      0.87       400
weighted avg       0.89      0.89      0.89       400


=== Best F1 (threshold = 0.31999999999999995) ===
[[257  21]
 [ 16 106]]
              precision    recall  f1-score   support

         0.0       0.94      0.92      0.93       278
         1.0       0.83      0.87      0.85       122

    accuracy                           0.91       400
   macro avg       0.89      0.90      0.89       400
weighted avg       0.91      0.91      0.91       400


=== Recall>=0.90 (threshold = 0.24999999999999994) ===
[[244  34]
 [ 12 110]]
              precision    recall  f1-score   support

         0.0       0.95      0.88      0.91       278
         1.0       0.